# Model Accuracy Report

Compares the prediction accuracy of each model, both over the full sample and year by year. Metrics reported per model: out-of-sample R², information coefficient (IC), rank IC, the IC information ratio, and error terms. The year-by-year breakdown shows whether a model's accuracy is stable or driven by a few strong years.

Input: the `*_predictions.parquet` files.


In [ ]:
import numpy as np
import pandas as pd
from pathlib import Path
import matplotlib.pyplot as plt

In [ ]:
# ---- configuration ---------------------------------------------------------
# each model just needs its predictions file (permno, eom, target_w, prediction).
MODELS = {
    'NN1':   'NN1_predictions.parquet',
    'NN2':   'NN2_predictions.parquet',
    'NN3':   'NN3_predictions.parquet',
    'NN4':   'NN4_predictions.parquet',
    'NN5':   'NN5_predictions.parquet',
    'RF':    'RF_predictions.parquet',
    'RIDGE': 'RIDGE_predictions.parquet',
    'KNN':   'KNN_predictions.parquet',
    'XGB':   'XGB_predictions.parquet',
}

FIRST_OOS_YEAR = 1994    # first out-of-sample test year

# keep only models whose predictions file exists
MODELS = {k: v for k, v in MODELS.items() if Path(v).exists()}
print('models with predictions:', list(MODELS))

## 1. Prediction accuracy (pooled)

R2_oos uses the Gu-Kelly-Xiu definition (sum of squared residuals against zero rather than the mean), so it is usually small or negative for monthly returns; treat it as a relative ranking, not an absolute fit. IC and rank IC (the correlation between prediction and realized return) are the more reliable measures of signal quality. IC_IR is the average monthly IC divided by its standard deviation.


In [ ]:
def accuracy_stats(df):
    d = df.dropna(subset=['prediction', 'target_w'])
    y = d['target_w'].to_numpy(); p = d['prediction'].to_numpy()
    ss_res = np.sum((y - p) ** 2)
    r2_oos = 1 - ss_res / np.sum(y ** 2)                    # GKX (vs zero)
    r2_dm  = 1 - ss_res / np.sum((y - y.mean()) ** 2)       # vs mean
    ic  = np.corrcoef(p, y)[0, 1]
    ric = pd.Series(p).corr(pd.Series(y), method='spearman')
    mic = d.groupby('eom').apply(
        lambda g: np.corrcoef(g['prediction'], g['target_w'])[0, 1]
                  if len(g) > 2 else np.nan).dropna()
    ic_ir = mic.mean() / mic.std() if mic.std() > 0 else np.nan
    ic_t  = ic_ir * np.sqrt(len(mic)) if len(mic) else np.nan
    return dict(R2_oos=r2_oos, R2_demean=r2_dm, IC=ic, RankIC=ric,
                IC_IR=ic_ir, IC_t=ic_t, MSE=np.mean((y-p)**2),
                MAE=np.mean(np.abs(y-p)), n=len(d))

PRED = {}
for k, path in MODELS.items():
    d = pd.read_parquet(path)
    d['eom'] = pd.to_datetime(d['eom'])
    PRED[k] = d

acc = pd.DataFrame({k: accuracy_stats(df) for k, df in PRED.items()}).T
acc = acc.sort_values('RankIC', ascending=False)
acc.round(4)

In [ ]:
# bar chart of Rank-IC (the headline signal-quality metric) by model
fig, ax = plt.subplots(figsize=(8, 4))
a = acc.sort_values('RankIC')
ax.barh(a.index, a['RankIC'], color='steelblue', edgecolor='black', linewidth=0.5)
ax.set_xlabel('Rank-IC (Spearman corr of prediction vs realized return)')
ax.set_title('Signal quality by model')
ax.axvline(0, color='k', lw=0.8); ax.grid(axis='x', alpha=0.3)
plt.tight_layout(); plt.show()

**Reading the pooled results.** Ranked by rank IC, the strongest model is ____ (rank IC ____), followed by ____ and ____. The weakest is ____ at ____. The spread between best and worst is ____, which is [large / modest], suggesting the choice of model [matters materially / makes little difference] for signal quality here. R2_oos is ____ for the top model; a [positive / near-zero / negative] value is consistent with the low predictability typical of monthly cross-sectional returns. Rank IC exceeding plain IC for a given model would indicate its edge is in *ordering* stocks rather than predicting return magnitudes, which is what matters for portfolio sorting.


## 2. Accuracy by year

The same metrics computed within each calendar year. The heatmap shows models on the vertical axis and years on the horizontal: a model whose row holds a steady color works consistently, while scattered bright cells indicate accuracy that may not persist out of sample.


In [ ]:
def yearly_accuracy(df):
    d = df.dropna(subset=['prediction', 'target_w']).copy()
    d['year'] = d['eom'].dt.year
    out = []
    for yr, g in d.groupby('year'):
        y = g['target_w'].to_numpy(); p = g['prediction'].to_numpy()
        # monthly rank-IC within the year
        mic = g.groupby('eom').apply(
            lambda x: x['prediction'].corr(x['target_w'], method='spearman')
                      if len(x) > 2 else np.nan).dropna()
        out.append(dict(year=yr,
                        RankIC=mic.mean(),
                        IC=np.corrcoef(p, y)[0, 1] if len(g) > 2 else np.nan,
                        R2_oos=1 - np.sum((y - p) ** 2) / np.sum(y ** 2),
                        MSE=np.mean((y - p) ** 2), n=len(g)))
    return pd.DataFrame(out).set_index('year')

yearly = {k: yearly_accuracy(df) for k, df in PRED.items()}
# show the best-overall model's yearly detail
_best = acc.index[0]
print(f'yearly accuracy detail for top model ({_best}):')
yearly[_best].round(4)

In [ ]:
# heatmaps: models x years, for Rank-IC and IC
def year_matrix(metric):
    return pd.DataFrame({k: ys[metric] for k, ys in yearly.items()}).T

for metric, label in [('RankIC', 'Rank-IC'), ('IC', 'IC')]:
    mat = year_matrix(metric)
    fig, ax = plt.subplots(figsize=(max(10, 0.32 * mat.shape[1] + 3), 0.5 * len(mat) + 1.5))
    vmax = np.nanpercentile(np.abs(mat.values), 95)
    im = ax.imshow(mat.values, aspect='auto', cmap='RdYlGn', vmin=-vmax, vmax=vmax)
    ax.set_xticks(range(mat.shape[1])); ax.set_xticklabels(mat.columns, rotation=90, fontsize=8)
    ax.set_yticks(range(len(mat))); ax.set_yticklabels(mat.index)
    fig.colorbar(im, label=label)
    ax.set_title(f'{label} by model and year')
    plt.tight_layout(); plt.show()

In [ ]:
# consistency summary: how often each model has a positive-Rank-IC year, and its spread
consistency = pd.DataFrame({
    k: {'mean_yr_RankIC': ys['RankIC'].mean(),
        'pos_RankIC_years_%': (ys['RankIC'] > 0).mean() * 100,
        'worst_yr_RankIC': ys['RankIC'].min(),
        'best_yr_RankIC': ys['RankIC'].max(),
        'yr_RankIC_std': ys['RankIC'].std()}
    for k, ys in yearly.items()}).T
consistency = consistency.sort_values('pos_RankIC_years_%', ascending=False)
print('Year-level consistency (higher pos-years % + lower std = more robust signal):')
consistency.round(4).sort_values(by='mean_yr_RankIC', ascending=False)

**Reading the year-by-year results.** In the heatmap, ____ shows the most consistent accuracy, holding positive rank IC in ____% of years. ____, by contrast, swings between strong and weak years, with its edge concentrated around ____. The consistency table ranks models by share of positive-rank-IC years: ____ leads at ____%, and its year-to-year standard deviation of ____ is [the lowest / middling], marking it the most stable signal. A model with high pooled accuracy but low year-consistency (compare its rank here against Section 1) would be a sign that a few years carry its full-sample number — a caution worth noting before trusting it forward.
